# LAB 25 — Алгоритми пошуку та хешування: Три Практичні Задачі

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Що ми закріплюємо?

Цей lab-нотебук містить три задачі різного рівня складності, що перевіряють розуміння лінійного пошуку, бінарного пошуку та хешування на практиці.

| Задача | Тема | Рівень |
|--------|------|--------|
| 1 | Система автодоповнення через `bisect` | Середній |
| 2 | Власна хеш-таблиця з open addressing | Складний |
| 3 | Кеш підрахунку частот (Word Frequency Cache) | Середній |

---


---

## Задача 1 — Система Автодоповнення (Рівень: Середній)

---

### Постановка задачі

Вам потрібно реалізувати клас `AutocompleteSystem`, що дозволяє:

1. **Ефективно додавати слова** до відсортованого словника
2. **Миттєво знаходити всі слова**, що починаються з заданого префіксу

**Вимоги до складності:**
- `add_word(word)` → $O(\log n)$ для пошуку позиції + $O(n)$ для вставки (в масив)
- `search(prefix)` → $O(\log n)$ для знаходження діапазону + $O(k)$ для збору результатів, де $k$ — кількість збігів

**Підказка:** Використайте `bisect.bisect_left` для знаходження позиції початку і кінця збігів.

**Очікувана поведінка:**
```
ac = AutocompleteSystem(["apple", "application", "apply", "banana", "band"])
ac.search("app")    → ["apple", "application", "apply"]
ac.search("ban")    → ["banana", "band"]
ac.search("xyz")    → []
ac.search("")       → всі слова
```

---

In [ ]:
import bisect
from typing import Optional

class AutocompleteSystem:
    """
    Ефективна система автодоповнення на основі відсортованого списку та bisect.
    Підтримує O(log n) пошук позиції + O(k) збирання результатів.
    """

    def __init__(self, words: list[str] = None):
        # TODO: ініціалізуйте внутрішній відсортований список
        # Підказка: відсортуйте вхідні слова при ініціалізації
        self._words: list[str] = []
        if words:
            pass  # TODO: додайте слова з сортуванням

    def add_word(self, word: str) -> None:
        """
        Вставляє слово у відсортований список, зберігаючи порядок.
        TODO: використайте bisect.insort для O(log n) пошуку позиції + вставки.
        """
        pass

    def search(self, prefix: str) -> list[str]:
        """
        Повертає всі слова, що починаються з prefix.

        TODO:
        1. Використайте bisect.bisect_left щоб знайти лівий край діапазону
        2. Ітеруйте від цього індексу, поки слово починається з prefix
        3. Поверніть зібрані збіги

        Підказка: слово починається з prefix якщо word.startswith(prefix)
        """
        pass

    def __len__(self) -> int:
        return len(self._words)

    def __repr__(self) -> str:
        return f"AutocompleteSystem({self._words[:5]}{'...' if len(self._words) > 5 else ''})"


# ── Тести ────────────────────────────────────────────────────────────────────
def run_tests():
    print("=== Тести AutocompleteSystem ===")
    print()

    ac = AutocompleteSystem(["apple", "application", "apply", "banana", "band", "can", "candy"])

    tests = [
        ("app",  ["apple", "application", "apply"]),
        ("ban",  ["banana", "band"]),
        ("can",  ["can", "candy"]),
        ("xyz",  []),
        ("",     ["apple", "application", "apply", "banana", "band", "can", "candy"]),
        ("appl", ["apple", "application", "apply"]),
    ]

    all_passed = True
    for prefix, expected in tests:
        result = ac.search(prefix)
        status = "✓" if result == expected else "✗"
        if result != expected:
            all_passed = False
        print(f"  [{status}] search('{prefix}') → {result}")
        if result != expected:
            print(f"       Очікувалось: {expected}")

    print()
    print(f"  Додаємо нові слова через add_word():")
    ac.add_word("application_v2")
    ac.add_word("banana_split")
    result = ac.search("ban")
    expected_new = ["banana", "banana_split", "band"]
    status = "✓" if result == expected_new else "✗"
    print(f"  [{status}] Після add: search('ban') → {result}")

    print()
    print(f"Всі базові тести: {'ПРОЙДЕНО ✓' if all_passed else 'Є ПОМИЛКИ ✗'}")


run_tests()


### Рішення (розгорніть після спроби)

<details>
<summary>▶ Показати рішення</summary>

```python
def __init__(self, words: list[str] = None):
    self._words: list[str] = sorted(words) if words else []

def add_word(self, word: str) -> None:
    bisect.insort(self._words, word)   # O(log n) пошук + O(n) зсув

def search(self, prefix: str) -> list[str]:
    if not prefix:
        return list(self._words)           # порожній prefix → всі слова
    
    # Знаходимо лівий край діапазону збігів
    start = bisect.bisect_left(self._words, prefix)
    
    results = []
    for i in range(start, len(self._words)):
        word = self._words[i]
        if word.startswith(prefix):
            results.append(word)
        elif word > prefix + "\xff":    # вийшли за межі алфавітного діапазону
            break
    return results
```

**Чому це ефективно?**
- `bisect_left` → $O(\log n)$ знаходить стартову позицію
- Ітерація зупиняється як тільки слова більше не починаються з prefix → $O(k)$ замість $O(n)$

</details>


---

## Задача 2 — Власна Хеш-Таблиця з Open Addressing (Рівень: Складний)

---

### Постановка задачі

Реалізуйте повноцінну хеш-таблицю `HashTable` з **відкритою адресацією та лінійним зондуванням**, яка:

1. Підтримує вставку, пошук та видалення за $O(1)$ у середньому
2. **Автоматично збільшується** (resize) коли load factor > 0.67
3. Правильно обробляє **видалені слоти** (використовує sentinel «DELETED»)

**Чому потрібен sentinel для видалення?**
При видаленні ми не можемо просто поставити `None` — це зламає пошук для ключів, що проходили через цей слот при вставці (зондування зупиниться передчасно). Замість цього використовуємо спеціальний маркер `DELETED`.

**Очікуваний API:**
```python
ht = HashTable()
ht.set("name", "Alice")   # вставка
ht.get("name")            # → "Alice"
ht.delete("name")         # видалення
ht.get("name")            # → None
len(ht)                   # → кількість живих елементів
```

---

In [ ]:
class HashTable:
    """
    Хеш-таблиця з відкритою адресацією (лінійне зондування).
    Автоматично збільшується при load factor > 0.67.
    Правильно обробляє видалення через sentinel DELETED.
    """

    _DELETED = object()   # унікальний sentinel для позначення видалених слотів

    def __init__(self, initial_capacity: int = 8):
        self._capacity = initial_capacity
        # Кожен слот: None (порожній) | _DELETED (видалено) | (key, value) (живий)
        self._buckets: list = [None] * self._capacity
        self._size = 0       # кількість ЖИВИХ елементів (не рахуємо DELETED)

    def _get_start_index(self, key) -> int:
        """Обчислює початковий індекс через hash % capacity."""
        return hash(key) % self._capacity

    def _probe(self, key, for_insert: bool = False) -> int:
        """
        Лінійне зондування: шукає слот для ключа.

        TODO:
        Якщо for_insert=False (пошук/видалення):
            - Зупиняємось на слоті де bucket[idx] == None (елемент точно відсутній)
            - Зупиняємось якщо знайшли (key, value) де key == шуканий ключ
            - Пропускаємо _DELETED слоти (продовжуємо зондування!)
            - Повертаємо індекс

        Якщо for_insert=True (вставка):
            - Зупиняємось на першому ВІЛЬНОМУ слоті: None або _DELETED
            - Але спочатку треба переконатись, що ключа ще немає (оновлення!)
            - Повертаємо індекс для вставки

        Підказка для лінійного зондування:
            idx = (idx + 1) % self._capacity
        """
        pass  # TODO

    def set(self, key, value) -> None:
        """
        Вставляє або оновлює (key, value).
        TODO:
        1. Знайдіть слот через _probe(key, for_insert=True)
        2. Якщо слот вже містить цей ключ → оновіть значення (не збільшуйте _size)
        3. Інакше → вставте новий елемент, збільште _size
        4. Якщо load_factor > 0.67 → викличте _resize()
        """
        pass  # TODO

    def get(self, key):
        """
        Повертає значення за ключем або None якщо ключа немає.
        TODO: знайдіть слот через _probe(key, for_insert=False)
        """
        pass  # TODO

    def delete(self, key) -> bool:
        """
        Видаляє ключ. Повертає True якщо знайдено і видалено, False якщо ні.
        TODO: замість None → поставте _DELETED sentinel!
        """
        pass  # TODO

    def _resize(self) -> None:
        """
        Збільшує capacity вдвічі і перехешовує всі живі елементи.
        TODO:
        1. Збережіть старі buckets
        2. Встановіть новий capacity = capacity * 2
        3. Скиньте _buckets на [None] * new_capacity та _size = 0
        4. Перевставте всі живі елементи через self.set()
        """
        pass  # TODO

    @property
    def load_factor(self) -> float:
        return self._size / self._capacity

    def __len__(self) -> int:
        return self._size

    def __repr__(self) -> str:
        items = {b[0]: b[1] for b in self._buckets if b and b is not self._DELETED}
        return f"HashTable({items}, capacity={self._capacity}, load={self.load_factor:.0%})"


# ── Тести ────────────────────────────────────────────────────────────────────
def run_hashtable_tests():
    print("=== Тести HashTable ===")
    print()

    ht = HashTable(initial_capacity=4)

    # Базова вставка та пошук
    ht.set("name", "Alice")
    ht.set("age", 30)
    ht.set("city", "Kyiv")
    print(f"Після 3 вставок: len={len(ht)}, load={ht.load_factor:.0%}")
    print(f"  get('name') = {ht.get('name')}  (очікується 'Alice')")
    print(f"  get('age')  = {ht.get('age')}   (очікується 30)")

    # Оновлення існуючого ключа
    ht.set("age", 31)
    print(f"  Оновлення 'age'=31: {ht.get('age')}  (очікується 31)")
    print(f"  len після оновлення: {len(ht)}  (очікується 3)")

    # Відсутній ключ
    print(f"  get('missing') = {ht.get('missing')}  (очікується None)")

    # Видалення
    print()
    deleted = ht.delete("age")
    print(f"  delete('age') = {deleted}  (очікується True)")
    print(f"  get('age') після видалення = {ht.get('age')}  (очікується None)")
    print(f"  len після видалення: {len(ht)}  (очікується 2)")

    # Подвійне видалення
    deleted2 = ht.delete("age")
    print(f"  delete('age') знову = {deleted2}  (очікується False)")

    # Авторозширення: додаємо ще елементів
    print()
    print("  Тест автозбільшення (resize):")
    ht2 = HashTable(initial_capacity=4)
    for i in range(10):
        ht2.set(f"key_{i}", i * 100)
    print(f"  Після 10 вставок: len={len(ht2)}, capacity={ht2._capacity}")
    print(f"  Всі значення доступні: {all(ht2.get(f'key_{i}') == i*100 for i in range(10))}")

    print()
    print("  Тест пошуку після видалення (DELETED sentinel):")
    ht3 = HashTable(initial_capacity=4)
    ht3.set("a", 1)
    ht3.set("b", 2)  # може потрапити в той самий bucket що і 'a', проходячи через нього
    ht3.delete("a")  # ставимо DELETED sentinel
    print(f"  get('b') після delete('a') = {ht3.get('b')}  (очікується 2, не None!)")


run_hashtable_tests()


### Рішення (розгорніть після спроби)

<details>
<summary>▶ Показати рішення</summary>

```python
def _probe(self, key, for_insert: bool = False) -> int:
    idx = self._get_start_index(key)
    first_deleted = None
    
    while True:
        bucket = self._buckets[idx]
        
        if bucket is None:
            # Порожній слот — кінець ланцюга пошуку
            if for_insert and first_deleted is not None:
                return first_deleted  # вставляємо у перший DELETED
            return idx
        
        if bucket is self._DELETED:
            if for_insert and first_deleted is None:
                first_deleted = idx   # запам'ятовуємо першый DELETED
        elif bucket[0] == key:
            return idx               # знайшли ключ
        
        idx = (idx + 1) % self._capacity

def set(self, key, value) -> None:
    idx = self._probe(key, for_insert=True)
    if self._buckets[idx] and self._buckets[idx] is not self._DELETED and self._buckets[idx][0] == key:
        self._buckets[idx] = (key, value)  # оновлення, без збільшення _size
    else:
        self._buckets[idx] = (key, value)
        self._size += 1
    if self.load_factor > 0.67:
        self._resize()

def get(self, key):
    idx = self._probe(key, for_insert=False)
    b = self._buckets[idx]
    if b is None or b is self._DELETED:
        return None
    return b[1]

def delete(self, key) -> bool:
    idx = self._probe(key, for_insert=False)
    b = self._buckets[idx]
    if b is None or b is self._DELETED:
        return False
    self._buckets[idx] = self._DELETED
    self._size -= 1
    return True

def _resize(self) -> None:
    old_buckets = self._buckets
    self._capacity *= 2
    self._buckets = [None] * self._capacity
    self._size = 0
    for b in old_buckets:
        if b and b is not self._DELETED:
            self.set(b[0], b[1])
```

</details>


---

## Задача 3 — Word Frequency Cache: Аналіз Тексту (Рівень: Середній)

---

### Постановка задачі

Реалізуйте клас `WordFrequencyAnalyzer`, що:

1. **Обробляє текст**: очищає від знаків пунктуації, нормалізує регістр
2. **Підраховує частоти** слів через `dict` (O(1) доступ)
3. **Знаходить топ-N** найпоширеніших слів
4. **Шукає слова за префіксом** через `bisect` на відсортованому словнику
5. **Порівнює ефективність** `dict` vs `list` для перевірки наявності

**Практичний контекст:**
Це реальна задача NLP (Natural Language Processing) та пошукових систем. Інвертований індекс (inverted index) пошукових рушіїв — це по суті `dict`, де ключ — слово, значення — список документів.

---

In [ ]:
import re
from collections import defaultdict
import bisect
import timeit

class WordFrequencyAnalyzer:
    """
    Аналізатор частот слів: демонструє O(1) dict проти O(n) list на практиці.
    """

    def __init__(self):
        self._freq: dict[str, int] = {}          # слово → кількість входжень
        self._sorted_vocab: list[str] = []       # відсортований словник для bisect

    def process_text(self, text: str) -> None:
        """
        Обробляє текст: токенізує, нормалізує, підраховує частоти.
        TODO:
        1. Переведіть текст у нижній регістр: text.lower()
        2. Витягніть слова через re.findall(r'[a-zа-яёіїєa-z\']+', text)
        3. Для кожного слова збільшіть лічильник у self._freq
        4. Перебудуйте self._sorted_vocab як відсортований список ключів _freq
        """
        pass  # TODO

    def top_n(self, n: int) -> list[tuple[str, int]]:
        """
        Повертає n найпоширеніших слів у форматі [(слово, кількість), ...]
        TODO: відсортуйте self._freq.items() за значенням (спадання), поверніть перші n
        """
        pass  # TODO

    def contains_dict(self, word: str) -> bool:
        """O(1) перевірка через dict."""
        pass  # TODO

    def contains_list(self, word: str) -> bool:
        """O(n) перевірка через пошук у списку (для порівняння)."""
        return word in list(self._freq.keys())

    def search_by_prefix(self, prefix: str) -> list[str]:
        """
        O(log n) пошук слів за префіксом через bisect у відсортованому словнику.
        TODO: використайте bisect.bisect_left + ітерацію зі startswith
        """
        pass  # TODO

    @property
    def vocab_size(self) -> int:
        return len(self._freq)

    def frequency(self, word: str) -> int:
        return self._freq.get(word, 0)


# ── Тексти для аналізу ────────────────────────────────────────────────────────
SAMPLE_TEXT = """
Python is an interpreted high-level general-purpose programming language.
Python's design philosophy emphasizes code readability with the use of significant indentation.
Python is dynamically typed and garbage-collected. It supports multiple programming paradigms,
including structured, object-oriented and functional programming. Python is often described
as a "batteries included" language due to its comprehensive standard library.
Python was designed by Guido van Rossum and first released in 1991.
Python consistently ranks as one of the most popular programming languages.
"""

# ── Тести ────────────────────────────────────────────────────────────────────
def run_analyzer_tests():
    print("=== Тести WordFrequencyAnalyzer ===")
    print()

    analyzer = WordFrequencyAnalyzer()
    analyzer.process_text(SAMPLE_TEXT)

    print(f"Розмір словника: {analyzer.vocab_size} унікальних слів")
    print()

    print("Топ-5 найпоширеніших слів:")
    for word, count in analyzer.top_n(5):
        print(f"  '{word}': {count} разів")
    print()

    test_words = ["python", "programming", "xyz_missing"]
    for w in test_words:
        print(f"  contains_dict('{w}') = {analyzer.contains_dict(w)}")
    print()

    print("Пошук за префіксом 'pro':")
    results = analyzer.search_by_prefix("pro")
    print(f"  {results}")
    print()

    # Порівняння продуктивності: dict O(1) vs list O(n)
    print("=== Порівняння продуктивності: dict vs list ===")
    N = 100_000
    big_text = (SAMPLE_TEXT * 1000)  # збільшуємо словник
    big_analyzer = WordFrequencyAnalyzer()
    big_analyzer.process_text(big_text)

    search_word = "python"

    t_dict = timeit.timeit(
        stmt='analyzer.contains_dict(word)',
        globals={'analyzer': big_analyzer, 'word': search_word},
        number=N
    ) / N * 1_000_000

    t_list = timeit.timeit(
        stmt='analyzer.contains_list(word)',
        globals={'analyzer': big_analyzer, 'word': search_word},
        number=1000
    ) / 1000 * 1_000_000

    print(f"  dict O(1): {t_dict:.3f} мкс")
    print(f"  list O(n): {t_list:.1f} мкс  (розмір словника: {big_analyzer.vocab_size})")
    print(f"  Прискорення dict: {t_list/t_dict:.1f}x")


run_analyzer_tests()


### Рішення (розгорніть після спроби)

<details>
<summary>▶ Показати рішення</summary>

```python
def process_text(self, text: str) -> None:
    words = re.findall(r"[a-zA-Zа-яА-ЯёЁіІїЇєЄ']+", text.lower())
    for word in words:
        self._freq[word] = self._freq.get(word, 0) + 1
    self._sorted_vocab = sorted(self._freq.keys())

def top_n(self, n: int) -> list[tuple[str, int]]:
    return sorted(self._freq.items(), key=lambda kv: kv[1], reverse=True)[:n]

def contains_dict(self, word: str) -> bool:
    return word in self._freq   # O(1) — хешовий пошук

def search_by_prefix(self, prefix: str) -> list[str]:
    start = bisect.bisect_left(self._sorted_vocab, prefix)
    results = []
    for i in range(start, len(self._sorted_vocab)):
        w = self._sorted_vocab[i]
        if w.startswith(prefix):
            results.append(w)
        elif w > prefix + "\xff":
            break
    return results
```

**Ключові архітектурні рішення:**
- `self._freq` (dict) → O(1) для всіх частотних операцій
- `self._sorted_vocab` (list) → підтримує bisect для O(log n) prefix search
- Два представлення одних даних → різні операції кожен ефективний по-своєму

</details>

---

## Підсумок Lab

| Задача | Алгоритми | Що перевіряли |
|--------|-----------|---------------|
| Автодоповнення | `bisect.insort`, `bisect_left` | Бінарний пошук на реальній задачі |
| Хеш-таблиця | Open addressing, linear probing, DELETED sentinel | Внутрішня механіка dict |
| Word Frequency | `dict` O(1), `list` O(n), bisect prefix | Вибір правильної структури |

> **Головний інсайт після lab:**
> Вибір структури даних визначає архітектуру. `dict` для доступу, `list` + `bisect` для пошуку по діапазону — це не взаємовиключні рішення, а інструменти для різних задач.
